In [1]:
import pandas as pd
import numpy as np

In [2]:
path = "assets/income.csv"

In [3]:
df = pd.read_csv(path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   education       32561 non-null  str  
 3   education-num   32561 non-null  int64
 4   marital-status  32561 non-null  str  
 5   occupation      30718 non-null  str  
 6   relationship    32561 non-null  str  
 7   race            32561 non-null  str  
 8   sex             32561 non-null  str  
 9   capital-gain    32561 non-null  int64
 10  capital-loss    32561 non-null  int64
 11  hours-per-week  32561 non-null  int64
 12  native-country  31978 non-null  str  
 13  income >50K     32561 non-null  int64
dtypes: int64(6), str(8)
memory usage: 3.5 MB


# Profile

In [4]:
df.describe()

,age,education-num,capital-gain,capital-loss,hours-per-week,income >50K
count,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,10.080679,1077.648844,87.303830,40.437456,0.240810
std,13.640433,2.572720,7385.292085,402.960219,12.347429,0.427581
min,17.000000,1.000000,0.000000,0.000000,1.000000,0.000000
25%,28.000000,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,10.000000,0.000000,0.000000,40.000000,0.000000
75%,48.000000,12.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,16.000000,99999.000000,4356.000000,99.000000,1.000000


In [5]:
df.duplicated().sum()

np.int64(3465)

In [6]:
df.isnull().sum()

age                  0
workclass         1836
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income >50K          0
dtype: int64

In [7]:
df.isnull().mean().sort_values(ascending=False).head()

occupation        0.056601
workclass         0.056386
native-country    0.017905
age               0.000000
education-num     0.000000
dtype: float64

## Cleaning

In [8]:
# df = df.dropna()
# df.isnull().sum()

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df = df.drop(columns=[c for c in df.columns if df[c].isnull().mean() > 0.6])
df.isnull().sum()

age                  0
workclass         1632
education            0
education-num        0
marital-status       0
occupation        1639
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     580
income >50K          0
dtype: int64

In [12]:
df.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income >50K
0,39,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


In [13]:
df.dropna(inplace=True)
df.isnull().sum()

age               0
workclass         0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income >50K       0
dtype: int64

### Handle outliers with IQR

In [15]:
for col in df.select_dtypes(include=np.number):
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5 * IQR
    higher_bound = Q3 + 1.5 * IQR
    mask = (df[col] >= lower_bound) & (df[col] <= higher_bound)
    df = df[mask]

In [16]:
df.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income >50K
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0
5,37,Private,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,0
13,32,Private,Assoc-acdm,12,Never-married,Sales,Not-in-family,Black,Male,0,0,50,United-States,0


### Type casting

In [19]:
for col in df.select_dtypes(include=["str"]):
    if df[col].nunique() < 20:
        df[col] = df[col].astype("category")


In [22]:
df.describe()

,age,education-num,capital-gain,capital-loss,hours-per-week,income >50K
count,12857.000000,12857.000000,12857.0,12857.0,12857.000000,12857.0
mean,37.316559,9.702963,0.0,0.0,41.235747,0.0
std,12.035203,2.391624,0.0,0.0,3.915592,0.0
min,17.000000,3.000000,0.0,0.0,33.000000,0.0
25%,27.000000,9.000000,0.0,0.0,40.000000,0.0
50%,35.000000,9.000000,0.0,0.0,40.000000,0.0
75%,46.000000,11.000000,0.0,0.0,40.000000,0.0
max,76.000000,16.000000,0.0,0.0,52.000000,0.0
